# Harmony default - mouse_mammary

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR


In [1]:
import gc
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import harmonypy as hm
import numpy as np
import pandas as pd
import scanpy as sc

from scib.metrics.metrics import metrics


In [2]:
DATASET_NAME = 'mouse_mammary'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'BATCH'
LABEL_KEY = 'celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_PCS = 50
N_NEIGHBORS = 15
N_TOP_HVG = 2000

EMBED_KEY = "X_harmony"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "harmony_metric_batch"
METRIC_LABEL_KEY = "harmony_metric_label"

assert DATASET_NAME == 'mouse_mammary'
assert BATCH_KEY == 'BATCH'
assert LABEL_KEY == 'celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
    },
    name="configuration",
)


dataset                                          mouse_mammary
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                BATCH
label_key                                             celltype
metrics               [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                           [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Name: configuration, dtype: object

In [3]:
def validate_dataset_contents():
    adata_backed = ad.read_h5ad(DATA_PATH, backed="r")
    try:
        required_keys = (BATCH_KEY, LABEL_KEY)
        missing_keys = [
            key for key in required_keys if key not in adata_backed.obs
        ]
        if missing_keys:
            raise ValueError(
                f"Missing obs keys {missing_keys}. "
                f"Available keys: {list(adata_backed.obs.columns)}"
            )

        obs = adata_backed.obs[[BATCH_KEY, LABEL_KEY]].copy()
        pseudotime = None
        if "TRAJ" in SELECTED_METRICS:
            if "dpt_pseudotime" not in adata_backed.obs:
                raise ValueError(
                    "TRAJ requires adata.obs['dpt_pseudotime']."
                )
            pseudotime = adata_backed.obs["dpt_pseudotime"].copy()

        n_obs = int(adata_backed.n_obs)
        n_vars = int(adata_backed.n_vars)
    finally:
        adata_backed.file.close()

    if obs.isna().any().any():
        raise ValueError(
            "Missing batch/label values: "
            f"{obs.isna().sum().to_dict()}"
        )

    n_batches = int(obs[BATCH_KEY].nunique())
    n_labels = int(obs[LABEL_KEY].nunique())
    if n_batches < 2:
        raise ValueError(
            f"Batch metrics require at least 2 batches; found {n_batches}."
        )

    batch_coverage = obs.groupby(
        LABEL_KEY,
        observed=True,
    )[BATCH_KEY].nunique()
    label_counts = obs.groupby(LABEL_KEY, observed=True).size()

    if "TRAJ" in SELECTED_METRICS:
        finite_pseudotime = pseudotime.dropna()
        if (
            len(finite_pseudotime) < 2
            or finite_pseudotime.nunique() < 2
        ):
            raise ValueError(
                "TRAJ requires at least 2 usable pseudotime values."
            )

    if "ISOLISI" in SELECTED_METRICS:
        min_batch_coverage = int(batch_coverage.min())
        if min_batch_coverage >= n_batches:
            raise ValueError(
                "ISOLISI is enabled, but every label appears in all batches."
            )

    eligible_cpcr_labels = [
        str(label)
        for label in label_counts.index
        if int(label_counts.loc[label]) >= 100
        and int(batch_coverage.loc[label]) >= 2
    ]
    if "CPCR" in SELECTED_METRICS and not eligible_cpcr_labels:
        raise ValueError(
            "CPCR requires a label with at least 100 cells across "
            "at least 2 batches."
        )

    return {
        "n_obs": n_obs,
        "n_vars": n_vars,
        "n_batches": n_batches,
        "n_labels": n_labels,
        "cpcr_eligible_labels": len(eligible_cpcr_labels),
    }


validate_dataset_contents()


{'n_obs': 9288,
 'n_vars': 1222,
 'n_batches': 3,
 'n_labels': 3,
 'cpcr_eligible_labels': 3}

In [4]:
HARMONY_DEFAULTS = {
    "theta": None,
    "lamb": None,
    "sigma": 0.1,
    "nclust": None,
    "tau": 0.0,
    "block_size": 0.05,
    "max_iter_harmony": 10,
    "max_iter_kmeans": 20,
    "epsilon_cluster": 1e-5,
    "epsilon_harmony": 1e-4,
    "alpha": 0.2,
}


def run_harmony_default(adata, seed):
    adata_int = adata.copy()

    if "highly_variable" in adata_int.var:
        adata_int = adata_int[:, adata_int.var["highly_variable"]].copy()
    else:
        sc.pp.highly_variable_genes(
            adata_int,
            flavor="seurat",
            n_top_genes=N_TOP_HVG,
        )
        adata_int = adata_int[:, adata_int.var["highly_variable"]].copy()

    sc.tl.pca(
        adata_int,
        n_comps=N_PCS,
        svd_solver="arpack",
    )
    x_pca = np.asarray(adata_int.obsm["X_pca"], dtype=np.float32)

    harmony_result = hm.run_harmony(
        x_pca,
        adata_int.obs,
        BATCH_KEY,
        **HARMONY_DEFAULTS,
        verbose=False,
        random_state=seed,
        device="cpu",
    )

    x_harmony = np.asarray(harmony_result.Z_corr)
    if (
        x_harmony.shape[0] != adata_int.n_obs
        and x_harmony.shape[1] == adata_int.n_obs
    ):
        x_harmony = x_harmony.T
    if x_harmony.shape[0] != adata_int.n_obs:
        raise ValueError(
            f"Harmony output shape {x_harmony.shape} is incompatible "
            f"with n_obs={adata_int.n_obs}."
        )
    if not np.all(np.isfinite(x_harmony)):
        raise ValueError("Harmony produced NaN or Inf in X_harmony.")

    adata_int.obsm[EMBED_KEY] = x_harmony
    return adata_int


pd.Series(HARMONY_DEFAULTS, name="upstream_default_argument")


theta                    NaN
lamb                     NaN
sigma                0.10000
nclust                   NaN
tau                  0.00000
block_size           0.05000
max_iter_harmony    10.00000
max_iter_kmeans     20.00000
epsilon_cluster      0.00001
epsilon_harmony      0.00010
alpha                0.20000
Name: upstream_default_argument, dtype: float64

In [5]:
def metric_value(metric_series, key):
    if key not in metric_series.index:
        raise KeyError(
            f"Metric {key} not found. "
            f"Available metrics: {list(metric_series.index)}"
        )

    value = metric_series[key]
    if isinstance(value, (list, tuple, np.ndarray)):
        values = np.asarray(value, dtype=float)
        if values.size == 0 or not np.all(np.isfinite(values)):
            raise ValueError(
                f"Metric {key} returned non-finite values: {values}"
            )
        return float(np.mean(values))

    value = float(value)
    if not np.isfinite(value):
        raise ValueError(
            f"Metric {key} returned a non-finite value: {value}"
        )
    return value


def compute_joint_metrics(adata_raw, adata_int):
    sc.pp.neighbors(
        adata_int,
        use_rep=EMBED_KEY,
        n_neighbors=N_NEIGHBORS,
        random_state=0,
    )

    metric_key_map = {
        METRIC_BATCH_KEY: BATCH_KEY,
        METRIC_LABEL_KEY: LABEL_KEY,
    }
    for adata_name, adata_obj in (
        ("adata_raw", adata_raw),
        ("adata_int", adata_int),
    ):
        existing_aliases = [
            alias for alias in metric_key_map if alias in adata_obj.obs
        ]
        if existing_aliases:
            raise ValueError(
                f"Reserved metric obs keys already exist in {adata_name}: "
                f"{existing_aliases}"
            )
        for alias, source_key in metric_key_map.items():
            adata_obj.obs[alias] = adata_obj.obs[source_key].copy()

    metric_df = metrics(
        adata=adata_raw,
        adata_int=adata_int,
        batch_key=METRIC_BATCH_KEY,
        label_key=METRIC_LABEL_KEY,
        embed=EMBED_KEY,
        cluster_key=CLUSTER_KEY,
        verbose=False,
        bami_="BAMI" in SELECTED_METRICS,
        casw_="CASW" in SELECTED_METRICS,
        scgraphm_="SCGRAPHM" in SELECTED_METRICS,
        traj_="TRAJ" in SELECTED_METRICS,
        isolisi_="ISOLISI" in SELECTED_METRICS,
        bras_="BRAS" in SELECTED_METRICS,
        kbet_="KBET" in SELECTED_METRICS,
        cpcr_="CPCR" in SELECTED_METRICS,
    )
    metric_series = metric_df.iloc[:, 0]

    result = {
        key: metric_value(metric_series, key)
        for key in SELECTED_METRICS
    }
    bio_score = float(
        np.mean([result[key] for key in SELECTED_BIO_METRICS])
    )
    batch_score = float(
        np.mean([result[key] for key in SELECTED_BATCH_METRICS])
    )
    joint_score = float(0.6 * bio_score + 0.4 * batch_score)

    if not np.all(np.isfinite([bio_score, batch_score, joint_score])):
        raise ValueError(
            "Aggregation produced a non-finite score: "
            f"bio={bio_score}, batch={batch_score}, "
            f"joint={joint_score}"
        )

    result["bio_score"] = bio_score
    result["batch_score"] = batch_score
    result["joint_score"] = joint_score
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_harmony_default(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


2026-07-21 15:14:15,413 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


2026-07-21 15:14:15,542 - harmonypy - INFO - KMeans initialization complete.


Cell type silhouette score...


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/sc

...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.5731957799334125


optimised clustering against harmony_metric_label
optimal cluster resolution: 0.4
optimal score: 0.9622571109563144
BRAS...


seed           9.000000
BAMI           0.962257
CASW           0.655185
SCGRAPHM       0.718199
BRAS           0.936662
KBET           0.075952
CPCR           0.996893
bio_score      0.778547
batch_score    0.669836
joint_score    0.735063
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.962806,0.655175,0.718195,0.936495,0.074528,0.996934,0.778725,0.669319,0.734963
1,1,0.960959,0.655181,0.718193,0.936628,0.075237,0.996995,0.778111,0.669620,0.734715
2,2,0.962709,0.655184,0.718194,0.936297,0.073943,0.996950,0.778696,0.669063,0.734843
3,3,0.962046,0.655188,0.718196,0.936435,0.074076,0.996940,0.778477,0.669150,0.734746
4,4,0.961498,0.655192,0.718194,0.936491,0.074526,0.996987,0.778295,0.669335,0.734711
5,5,0.962806,0.655202,0.718199,0.936490,0.075373,0.996939,0.778736,0.669600,0.735082
6,6,0.960427,0.655193,0.718197,0.936358,0.073918,0.996951,0.777939,0.669076,0.734394
7,7,0.962138,0.655177,0.718193,0.936523,0.074972,0.996973,0.778503,0.669490,0.734897
8,8,0.960962,0.655144,0.718192,0.936771,0.075260,0.997047,0.778100,0.669693,0.734737
9,9,0.962257,0.655185,0.718199,0.936662,0.075952,0.996893,0.778547,0.669836,0.735063


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.961861,0.000811,0.960427,0.962806
CASW,0.655182,0.000015,0.655144,0.655202
SCGRAPHM,0.718195,0.000002,0.718192,0.718199
BRAS,0.936515,0.000134,0.936297,0.936771
KBET,0.074779,0.000653,0.073918,0.075952
CPCR,0.996961,0.000040,0.996893,0.997047
bio_score,0.778413,0.000272,0.777939,0.778736
batch_score,0.669418,0.000257,0.669063,0.669836
joint_score,0.734815,0.000193,0.734394,0.735082
